## OpenAI

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
GLM_API_KEY=os.getenv('ZAI_API_KEY')

In [ ]:
client = OpenAI(
    api_key=GLM_API_KEY,
    base_url="https://api.z.ai/api/coding/paas/v4"
)

response = client.chat.completions.create(
    model="glm-5.1",
    messages=[
        {"role": "user", "content": "What is the capital of Bangladesh?"}
    ],
    stream=False
)

In [ ]:
print(response)

In [ ]:
import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

## Ollama

In [ ]:
!pip install ollama

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(
    model="llama3.1:8b",
    messages=[
        { 'role': 'user', 'content': 'Capital of Bangladesh?', },
    ]
)

print(response.message.content)

In [ ]:
import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

##

## Lite LLM

In [37]:
from litellm import completion

response = completion(
    model="ollama/llama3.1:8b",
    messages=[{"content": "What is the capital of Bangladesh", "role": "user"}],
    api_base="http://localhost:11434"
)

import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-ee8913f3-c2be-46e0-bac1-8dcd8193edde",
  "created": 1778865887,
  "model": "ollama/llama3.1:8b",
  "object": "chat.completion",
  "system_fingerprint": null,
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "The capital of Bangladesh is Dhaka.",
        "role": "assistant",
        "tool_calls": null,
        "function_call": null,
        "reasoning_content": null
      }
    }
  ],
  "usage": {
    "completion_tokens": 9,
    "prompt_tokens": 20,
    "total_tokens": 29,
    "completion_tokens_details": null,
    "prompt_tokens_details": null
  }
}


## Testing MCP

In [69]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-my-super-secret-key",
    base_url="http://localhost:4000"
)

response = client.chat.completions.create(
    model="glm-5.1",

    messages=[
        {"role": "user", "content": "Current temperature of dhaka?"}
    ],

    tools=[
        {
            "type": "mcp",
            "server_label": "litellm",
            "server_url": "litellm_proxy/mcp",
            "require_approval": "never"
        }
    ],
    stream=False
)


import json
print(json.dumps(response.model_dump(), indent=4))

{
    "id": "20260516022208d7d8d8d095004ff3",
    "choices": [
        {
            "finish_reason": "stop",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": "Here's the current weather information for **Dhaka, Bangladesh**:\n\n- \ud83c\udf21\ufe0f **Temperature:** 27.1\u00b0C\n- \ud83c\udf24\ufe0f **Condition:** Clear sky\n- \ud83d\udca8 **Wind Speed:** 11.3 km/h\n- \ud83d\udd50 **Time:** 6:15 PM (May 15, 2026)",
                "refusal": null,
                "role": "assistant",
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": null,
                "reasoning_content": "I have the weather data for Dhaka. Let me present this information to the user.",
                "provider_specific_fields": {
                    "refusal": null,
                    "reasoning_content": "I have the weather data for Dhaka. Let me present this information

In [80]:
from fastmcp import Client
import asyncio

config = {
    "mcpServers": {
        "petstore": {
            "url": "http://localhost:8000/mcp"
        }
    }
}

client = Client(config)


async def main():
    async with client:

        tools = await client.list_tools()
        print("Available tools:", [t.name for t in tools])

        # IMPORTANT: use real tool name from list_tools()
        response = await client.call_tool(
            name="get_weather",   # or whatever shows in list_tools()
            arguments={"city": "Dhaka"}  # match your actual tool schema
        )

        print("Response:", response)


await main()

Available tools: ['get_weather', 'get_time', 'calculate']
Response: CallToolResult(content=[TextContent(type='text', text='{"city":"Dhaka","country":"Bangladesh","temperature_c":27.0,"windspeed_kmh":9.8,"condition":"Clear sky","time":"2026-05-15T18:30"}', annotations=None, meta=None)], structured_content={'city': 'Dhaka', 'country': 'Bangladesh', 'temperature_c': 27.0, 'windspeed_kmh': 9.8, 'condition': 'Clear sky', 'time': '2026-05-15T18:30'}, meta=None, data={'city': 'Dhaka', 'country': 'Bangladesh', 'temperature_c': 27.0, 'windspeed_kmh': 9.8, 'condition': 'Clear sky', 'time': '2026-05-15T18:30'}, is_error=False)
